# Hafta 14 - Gemini API Temelleri

Bu defterde Google Gemini API'yi kullanarak yapay zeka uygulamaları geliştirmeyi öğreneceğiz.

## Öğrenme Hedefleri
- Gemini API kurulumu ve yapılandırması
- Metin üretimi (text generation)
- Çok turlu sohbet (multi-turn conversation)
- Sistem talimatları ve kişilik ayarlama
- Parametre ayarları (temperature, top_p)
- Yapılandırılmış çıktı (JSON)
- Çok modlu (multimodal) kullanım
- Token sayımı

## 1. Kurulum

Öncelikle `google-generativeai` paketini yükleyelim:

In [ ]:
!pip install -q google-generativeai Pillow

## 2. API Anahtarı Yapılandırması

> **Not:** API anahtarınızı [Google AI Studio](https://aistudio.google.com) adresinden ücretsiz alabilirsiniz.
> 1. Google hesabınızla giriş yapın
> 2. Sol menüden **"Get API Key"** seçin
> 3. **"Create API Key"** butonuna tıklayın
> 4. Anahtarı kopyalayıp aşağıdaki `YOUR_API_KEY` yerine yapıştırın

⚠️ **Güvenlik Uyarısı:** API anahtarınızı asla GitHub'a veya herkese açık yerlere yüklemeyin!

In [ ]:
import google.generativeai as genai

# API anahtarınızı buraya girin
API_KEY = "YOUR_API_KEY"  # <-- Kendi anahtarınızı yazın

genai.configure(api_key=API_KEY)

print("Gemini API başarıyla yapılandırıldı!")

## 3. Temel Metin Üretimi

Gemini modeli ile basit bir metin üretelim:

In [ ]:
# Model oluşturma
model = genai.GenerativeModel('gemini-2.5-flash')

# Basit metin üretimi
response = model.generate_content("Merhaba, kendini tanıt")

print(response.text)

### Gemini API Kullanımı

Google Gemini modeli ile metin üretimi yapıyoruz. Model, verilen prompt'a göre insan benzeri yanıtlar oluşturur.

In [ ]:
# Daha karmaşık bir istek
response = model.generate_content(
    "Yapay zekanın eğitimdeki 5 önemli kullanım alanını kısaca açıkla."
)

print(response.text)

## 4. Çok Turlu Sohbet (Multi-turn Conversation)

Gemini ile bağlamı koruyan bir sohbet başlatabiliriz. Model önceki mesajları hatırlar.

In [ ]:
# Sohbet başlatma
chat = model.start_chat(history=[])

# İlk mesaj
response = chat.send_message("Merhaba! Ben veri bilimi öğreniyorum.")
print("Gemini:", response.text)
print("---")

# İkinci mesaj - model bağlamı hatırlar
response = chat.send_message("Bana hangi programlama dilini öğrenmemi önerirsin?")
print("Gemini:", response.text)
print("---")

# Üçüncü mesaj
response = chat.send_message("Bu dilde ilk projeme ne önerirsin?")
print("Gemini:", response.text)

### Sohbet geçmişini görüntüleme

Aşağıdaki kod bloğunda bu işlemi gerçekleştiriyoruz.

In [ ]:
# Sohbet geçmişini görüntüleme
for message in chat.history:
    role = "Kullanıcı" if message.role == "user" else "Gemini"
    print(f"{role}: {message.parts[0].text[:100]}...")
    print()

## 5. Sistem Talimatı ve Kişilik Ayarlama (System Instruction)

`system_instruction` parametresi ile modele bir kişilik veya rol verebiliriz. Bu talimat her yanıtta etkili olur.

In [ ]:
# Türk mutfağı uzmanı olarak yapılandırma
chef_model = genai.GenerativeModel(
    'gemini-2.5-flash',
    system_instruction="""Sen deneyimli bir Türk şefisin. Adın Şef Ahmet.
    Türk mutfağı konusunda uzmansın. Yanıtlarını sıcak ve samimi bir dille ver.
    Tarifleri adım adım anlat. Her zaman malzeme listesi ver.
    Yemek kültürü hakkında ilginç bilgiler paylaş."""
)

response = chef_model.generate_content("Mantı nasıl yapılır?")
print(response.text)

### Gemini API Kullanımı

Google Gemini modeli ile metin üretimi yapıyoruz. Model, verilen prompt'a göre insan benzeri yanıtlar oluşturur.

In [ ]:
# Matematik öğretmeni olarak yapılandırma
math_model = genai.GenerativeModel(
    'gemini-2.5-flash',
    system_instruction="""Sen sabırlı bir matematik öğretmenisin.
    Konuları basit ve anlaşılır şekilde anlat.
    Her konuyu günlük hayattan örneklerle açıkla.
    Adım adım çözüm göster.
    Öğrenciyi motive edici bir dil kullan."""
)

response = math_model.generate_content("Türev nedir? Neden önemlidir?")
print(response.text)

## 6. Parametre Ayarları: Temperature ve Top-p

| Parametre | Açıklama | Aralık | Düşük Değer | Yüksek Değer |
|-----------|----------|--------|-------------|---------------|
| **temperature** | Yanıtın yaratıcılık seviyesi | 0.0 - 2.0 | Tutarlı, öngörülebilir | Yaratıcı, çeşitli |
| **top_p** | Olasılık eşiği (nucleus sampling) | 0.0 - 1.0 | Dar seçim | Geniş seçim |
| **top_k** | En olası k token arasından seçim | 1 - 100 | Dar seçim | Geniş seçim |
| **max_output_tokens** | Maksimum çıktı uzunluğu | 1 - 8192 | Kısa yanıt | Uzun yanıt |

In [ ]:
# Düşük temperature: tutarlı ve öngörülebilir yanıtlar
generation_config_low = genai.GenerationConfig(
    temperature=0.1,
    top_p=0.8,
    max_output_tokens=200
)

response_low = model.generate_content(
    "Bir hikaye yaz: Uzayda kaybolmuş bir astronot...",
    generation_config=generation_config_low
)
print("=== DÜŞÜK TEMPERATURE (0.1) ===")
print(response_low.text)
print()

### Gemini API Kullanımı

Google Gemini modeli ile metin üretimi yapıyoruz. Model, verilen prompt'a göre insan benzeri yanıtlar oluşturur.

In [ ]:
# Yüksek temperature: yaratıcı ve çeşitli yanıtlar
generation_config_high = genai.GenerationConfig(
    temperature=1.5,
    top_p=0.95,
    max_output_tokens=200
)

response_high = model.generate_content(
    "Bir hikaye yaz: Uzayda kaybolmuş bir astronot...",
    generation_config=generation_config_high
)
print("=== YÜKSEK TEMPERATURE (1.5) ===")
print(response_high.text)

## 7. Yapılandırılmış Çıktı (JSON)

Gemini'den JSON formatında yanıt isteyebiliriz. Bu, verileri programatik olarak işlemek için çok kullanışlıdır.

In [ ]:
import json

# JSON formatında çıktı isteme
json_prompt = """Aşağıdaki bilgileri JSON formatında ver:

Türkiye'nin en büyük 5 şehrini listele. Her şehir için şunları ekle:
- isim: Şehir adı
- nufus: Yaklaşık nüfus (milyon)
- bolge: Bulunduğu bölge
- ozellik: Bir cümlelik özellik

Sadece JSON döndür, başka açıklama ekleme.
JSON formatı: {"sehirler": [{"isim": ..., "nufus": ..., "bolge": ..., "ozellik": ...}]}
"""

response = model.generate_content(json_prompt)
print("Ham yanıt:")
print(response.text)
print()

### JSON'u parse etme

Aşağıdaki kod bloğunda bu işlemi gerçekleştiriyoruz.

In [ ]:
# JSON'u parse etme
try:
    # Bazen model ```json ... ``` ile sarar, temizleyelim
    json_text = response.text.strip()
    if json_text.startswith("```"):
        json_text = json_text.split("\n", 1)[1]  # İlk satırı atla
        json_text = json_text.rsplit("```", 1)[0]  # Son ```'ı kaldır
    
    data = json.loads(json_text)
    
    print("Parse edilmiş veriler:")
    for sehir in data["sehirler"]:
        print(f"  {sehir['isim']}: {sehir['nufus']}M - {sehir['ozellik']}")
except json.JSONDecodeError as e:
    print(f"JSON parse hatası: {e}")

## 8. Çok Modlu Kullanım (Multimodal): Görüntü Analizi

Gemini, metin ve görüntüyü birlikte işleyebilen çok modlu (multimodal) bir modeldir.

In [ ]:
import PIL.Image
import urllib.request
import os

# Örnek bir görüntü indirelim
image_url = "https://upload.wikimedia.org/wikipedia/commons/thumb/4/4b/Ulus_-_Ankara.jpg/640px-Ulus_-_Ankara.jpg"
image_path = "ornek_gorsel.jpg"

if not os.path.exists(image_path):
    urllib.request.urlretrieve(image_url, image_path)
    print("Görsel indirildi!")

# Görseli yükle
img = PIL.Image.open(image_path)
img

### Gemini API Kullanımı

Google Gemini modeli ile metin üretimi yapıyoruz. Model, verilen prompt'a göre insan benzeri yanıtlar oluşturur.

In [ ]:
# Görüntü hakkında soru sorma
response = model.generate_content(
    ["Bu fotoğrafı detaylı olarak açıkla. Nerede çekilmiş olabilir?", img]
)

print(response.text)

### Gemini API Kullanımı

Google Gemini modeli ile metin üretimi yapıyoruz. Model, verilen prompt'a göre insan benzeri yanıtlar oluşturur.

In [ ]:
# Görüntü hakkında yapılandırılmış analiz
response = model.generate_content(
    ["""Bu görseli analiz et ve JSON formatında şu bilgileri ver:
    - konum: Tahmini konum
    - nesneler: Görseldeki ana nesneler (liste)
    - ruh_hali: Görselin verdiği duygu
    - mevsim: Tahmini mevsim
    Sadece JSON döndür.""", img]
)

print(response.text)

## 9. Token Sayımı

Token, modelin metni işleme birimi dir. Maliyet ve limit kontrolü için token sayısını bilmek önemlidir.

- Yaklaşık olarak 1 Türkçe kelime ≈ 2-3 token
- Gemini 2.5 Flash: 1M token bağlam penceresi

In [ ]:
# Metin için token sayımı
metin = "Yapay zeka, günümüzde eğitimden sağlığa birçok alanda devrim yaratmaktadır."

token_count = model.count_tokens(metin)
print(f"Metin: {metin}")
print(f"Token sayısı: {token_count.total_tokens}")
print()

### Sohbet geçmişinin token sayısı

Aşağıdaki kod bloğunda bu işlemi gerçekleştiriyoruz.

In [ ]:
# Sohbet geçmişinin token sayısı
chat_token_count = model.count_tokens(chat.history)
print(f"Sohbet geçmişi token sayısı: {chat_token_count.total_tokens}")

### Uzun bir metin ile karşılaştırma

Aşağıdaki kod bloğunda bu işlemi gerçekleştiriyoruz.

In [ ]:
# Uzun bir metin ile karşılaştırma
kisa_metin = "Merhaba dünya"
uzun_metin = "Yapay zeka ve makine öğrenmesi, modern dünyada " * 50

print(f"Kısa metin token: {model.count_tokens(kisa_metin).total_tokens}")
print(f"Uzun metin token: {model.count_tokens(uzun_metin).total_tokens}")
print(f"\nUzun metin karakter sayısı: {len(uzun_metin)}")

## Özet

Bu defterde öğrendiklerimiz:

| Konu | Açıklama |
|------|----------|
| **API Kurulumu** | `google-generativeai` paketi ve API anahtarı yapılandırması |
| **Metin Üretimi** | `generate_content()` ile basit metin üretimi |
| **Çok Turlu Sohbet** | `start_chat()` ile bağlamı koruyan sohbet |
| **Sistem Talimatı** | `system_instruction` ile kişilik ve rol atama |
| **Parametreler** | `temperature`, `top_p`, `max_output_tokens` ayarları |
| **JSON Çıktı** | Yapılandırılmış veri formatında yanıt alma |
| **Multimodal** | Görüntü + metin birlikte işleme |
| **Token Sayımı** | `count_tokens()` ile maliyet ve limit kontrolü |

### Sonraki Adım
Bir sonraki defterde bu bilgileri kullanarak **kişisel asistan chatbot** oluşturacağız!